# Publication notebook: chapter-ready pair-noq figures and tables

- Purpose: export chapter-ready classical-model tables and comparison figures.
- Required inputs: retained classical metrics and pair-noq processed TSVs.
- Required models: tuned classical `.joblib` artifacts for ROC and PR analyses.
- Required external tools: none.
- Expected outputs: LaTeX tables and figures under `results/`.
- Publication output: chapter/table-ready classical comparison artifacts.
- Reproduction status: normalized to canonical paths; some cells remain dependent on local tuned model binaries.


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
metrics = pd.read_csv("../results/metrics/classical/baseline_pair_noq/metrics_summary.tsv", sep="\t")

cols = [
    "model",
    "test_accuracy",
    "test_precision",
    "test_recall",
    "test_f1",
    "test_roc_auc",
]

tab_baseline = metrics[cols].copy().round(3)

Path("../results/tables/notebook_exports").mkdir(parents=True, exist_ok=True)

tab_baseline.to_latex(
    "../results/tables/notebook_exports/baseline_models_pair_noq.tex",
    index=False,
    caption="Performance of baseline classifiers on the held-out test set.",
    label="tab:baseline_models_pair_noq",
    escape=False,
)

tab_baseline

In [ ]:
tuned = pd.read_csv("../results/metrics/classical/tuned_pair_noq/tuned_models_summary.tsv", sep="\t")

cols_tuned = [
    "model",
    "test_accuracy",
    "test_precision",
    "test_recall",
    "test_f1",
    "test_roc_auc",
]

tab_tuned = tuned[cols_tuned].copy().round(3)

Path("../results/tables/notebook_exports").mkdir(parents=True, exist_ok=True)

tab_tuned.to_latex(
    "../results/tables/notebook_exports/tuned_models_pair_noq.tex",
    index=False,
    caption="Performance of tuned classifiers on the held-out test set.",
    label="tab:tuned_models_pair_noq",
    escape=False,
)

tab_tuned

In [ ]:
Path("../results/figures/notebook_exports").mkdir(parents=True, exist_ok=True)

metrics_sorted = metrics.sort_values("test_f1", ascending=False)

plt.figure(figsize=(7, 4))
plt.barh(metrics_sorted["model"], metrics_sorted["test_f1"])
plt.xlabel("Test F1")
plt.title("Baseline models")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("../results/figures/notebook_exports/baseline_f1_pair.png", dpi=300, bbox_inches="tight")
plt.close()

print("Saved: ../results/figures/notebook_exports/baseline_f1_pair.png")

In [ ]:
baseline_path = Path("../results/metrics/classical/baseline_pair_noq/metrics_summary.tsv")
tuned_path = Path("../results/metrics/classical/tuned_pair_noq/tuned_models_summary.tsv")

baseline = pd.read_csv(baseline_path, sep="\t")
tuned = pd.read_csv(tuned_path, sep="\t")

for df_ in (baseline, tuned):
    if "model_base" in df_.columns:
        del df_["model_base"]

tuned["model_base"] = tuned["model"].str.replace("_tuned$", "", regex=True)
baseline["model_base"] = baseline["model"]

for df_ in (baseline, tuned):
    if "model" in df_.columns:
        del df_["model"]

merged = baseline.merge(
    tuned,
    on="model_base",
    suffixes=("_base", "_tuned"),
)

merged["model_name"] = merged["model_base"]
merged = merged.sort_values("test_f1_tuned", ascending=False).reset_index(drop=True)

plot_df = merged.copy()
plot_df

In [ ]:
x = range(len(plot_df))

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)

axes[0].bar(x, plot_df["test_f1_base"], label="baseline")
axes[0].bar(x, plot_df["test_f1_tuned"], alpha=0.7, label="tuned")
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(plot_df["model_name"], rotation=45, ha="right")
axes[0].set_ylabel("Test F1")
axes[0].set_title("F1 before vs after tuning")
axes[0].legend()

axes[1].bar(x, plot_df["test_roc_auc_base"], label="baseline")
axes[1].bar(x, plot_df["test_roc_auc_tuned"], alpha=0.7, label="tuned")
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(plot_df["model_name"], rotation=45, ha="right")
axes[1].set_ylabel("Test ROC-AUC")
axes[1].set_title("ROC-AUC before vs after tuning")
axes[1].legend()

plt.tight_layout()
plt.savefig("../results/figures/notebook_exports/f1_auc_tuning_pair.png", dpi=300, bbox_inches="tight")
plt.close()

print("Saved: ../results/figures/notebook_exports/f1_auc_tuning_pair.png")

In [ ]:
from joblib import load
from sklearn.metrics import (
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score,
)
from sklearn.impute import SimpleImputer
import numpy as np

def load_pair_noq_like_hparam_script(path: str):
    df = pd.read_csv(path, sep="\t")
    if "label" not in df.columns:
        raise ValueError("Expected a 'label' column in the dataset.")

    if "strand" in df.columns:
        df["strand"] = df["strand"].map({"+": 1, "-": 0})

    drop_cols = ["read_id", "ref_name", "cigar"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

    for c in df.columns:
        if c != "label":
            df[c] = pd.to_numeric(df[c], errors="coerce")

    feature_cols = [c for c in df.columns if c != "label"]
    X = df[feature_cols].to_numpy(dtype=float)
    y = df["label"].to_numpy(dtype=int)

    imp = SimpleImputer(strategy="median")
    X = imp.fit_transform(X)

    return X, y, feature_cols

X_test, y_test, feature_names = load_pair_noq_like_hparam_script("../data/processed/PAIR_test_noq.tsv")

models = {
    "catboost": load("../models/pair_noq_tuned/catboost_tuned.joblib"),
    "gradient_boosting": load("../models/pair_noq_tuned/gradient_boosting_tuned.joblib"),
    "bagging_trees": load("../models/pair_noq_tuned/bagging_trees_tuned.joblib"),
    "mlp": load("../models/pair_noq_tuned/mlp_tuned.joblib"),
}

In [ ]:
import sys, sklearn, joblib
print(sys.version)
print(sklearn.__version__)
print(joblib.__version__)

In [ ]:
Path("../results/figures/notebook_exports").mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(7, 5))
for name, model in models.items():
    scores = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, scores)
    auc = roc_auc_score(y_test, scores)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves (PAIR)")
plt.legend()
plt.tight_layout()
plt.savefig("../results/figures/notebook_exports/roc_curves_pair.png", dpi=300, bbox_inches="tight")
plt.close()

plt.figure(figsize=(7, 5))
for name, model in models.items():
    scores = model.predict_proba(X_test)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, scores)
    ap = average_precision_score(y_test, scores)
    plt.plot(rec, prec, label=f"{name} (AP={ap:.3f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall curves (PAIR)")
plt.legend()
plt.tight_layout()
plt.savefig("../results/figures/notebook_exports/pr_curves_pair.png", dpi=300, bbox_inches="tight")
plt.close()

print("Saved: ../results/figures/notebook_exports/roc_curves_pair.png")
print("Saved: ../results/figures/notebook_exports/pr_curves_pair.png")

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

def plot_cm_save(cm, title, out_path):
    plt.figure(figsize=(4.5, 4.0))
    plt.imshow(cm, cmap="Greys")
    plt.title(title)
    plt.xticks([0, 1], ["clean", "chimeric"])
    plt.yticks([0, 1], ["clean", "chimeric"])
    plt.xlabel("Predicted")
    plt.ylabel("True")

    threshold = cm.max() / 2.0
    for (i, j), v in np.ndenumerate(cm):
        text_color = "white" if v > threshold else "black"
        plt.text(j, i, str(v), ha="center", va="center", color=text_color)

    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_pred = np.asarray(y_pred).reshape(-1).astype(int)

    cm = confusion_matrix(y_test, y_pred)
    out_path = Path("figures") / f"cm_pair_{name}.png"
    plot_cm_save(cm, f"Confusion Matrix – {name} (PAIR)", out_path)
    print("Saved:", out_path)

In [ ]:
import json
import pandas as pd
from pathlib import Path

df = pd.read_csv("../data/processed/PAIR_train_noq.tsv", sep="\t")

if "strand" in df.columns:
    df["strand"] = df["strand"].map({"+": 1, "-": 0})

drop_cols = ["read_id", "ref_name", "cigar"]
df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

for c in df.columns:
    if c != "label":
        df[c] = pd.to_numeric(df[c], errors="coerce")

feature_cols = [c for c in df.columns if c != "label"]

out = Path("../models/pair_noq_tuned/feature_cols_24.json")
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(feature_cols, indent=2) + "\n")

print("Saved:", out)
print("n_features:", len(feature_cols))
print(feature_cols)